# eRetail AI Intelligence Platform
## Notebook 04: Heuristic & Statistical Forecasting Baselines

**Objective**: Benchmark heuristic and classical statistical forecasting baselines on regularized retail demand series, investigate the impact of stockout demand masking, and verify baseline behavior on regular vs intermittent series.

### Key Baselines Implemented:
1. **Naive Model**: Last observed unconstrained demand projected forward.
2. **Seasonal Naive Model**: 7-day cyclical projection capturing weekly retail cadence.
3. **Moving Average Model**: Rolling window mean (7d, 14d, 30d) smoothing high-frequency noise.
4. **Exponential Smoothing (Holt-Winters)**: Level, trend, and seasonal components with prediction intervals.
5. **Croston & SBA**: Intermittent demand modeling separating demand occurrence interval from size.


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

workspace_dir = Path.cwd().resolve()
if workspace_dir.name == "notebooks":
    workspace_dir = workspace_dir.parent
sys.path.insert(0, str(workspace_dir / "src"))

mpl_cache = workspace_dir / ".gemini" / "scratch" / "mpl"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_cache)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")
print("Environment configured successfully. Workspace:", workspace_dir)


Environment configured successfully. Workspace: /Users/avijeet/Documents/eRetail AI Intelligence


In [2]:
from commerce_ai.data.loaders import CommerceDataLoader
from commerce_ai.analytics.demand import build_daily_demand
from commerce_ai.analytics.stockout import mask_stockout_demand

data_dir = workspace_dir / "data" / "sample"
loader = CommerceDataLoader()
sales_df = loader.load_csv(data_dir / "sales.csv", validate=False)
inventory_df = loader.load_csv(data_dir / "inventory.csv", validate=False)
print(f"Loaded {len(sales_df):,} sales records, {len(inventory_df):,} inventory snapshots.")

# Select active SKUs for fast, focused benchmark
sample_skus = sales_df["sku_id"].value_counts().head(25).index.tolist()
daily_demand = build_daily_demand(
    sales=sales_df,
    inventory=inventory_df,
    skus=sample_skus,
    warehouse_ids=["WH_NORTH_01"],
)
demand_with_mask = mask_stockout_demand(daily_demand)
print(f"Constructed regularized demand grid: {len(demand_with_mask):,} rows across {demand_with_mask['sku_id'].nunique()} SKUs in WH_NORTH_01.")


Loaded 773,555 sales records, 292,500 inventory snapshots.
Constructed regularized demand grid: 18,250 rows across 25 SKUs in WH_NORTH_01.


In [3]:
from commerce_ai.analytics.features import calculate_intermittency_metrics

intermittency = calculate_intermittency_metrics(demand_with_mask)
print("Demand Intermittency Distribution across sample series:")
print(intermittency["intermittency_class"].value_counts())

# Select a regular (smooth) SKU and an intermittent SKU
smooth_skus = intermittency[intermittency["intermittency_class"] == "smooth"]["sku_id"]
regular_sku = smooth_skus.iloc[0] if not smooth_skus.empty else sample_skus[0]
target_wh = "WH_NORTH_01"

series_regular = demand_with_mask[
    (demand_with_mask["sku_id"] == regular_sku) & 
    (demand_with_mask["warehouse_id"] == target_wh)
].sort_values("date").reset_index(drop=True)

print(f"Selected regular SKU: {regular_sku} with {len(series_regular)} daily records.")


Demand Intermittency Distribution across sample series:
intermittency_class
Regular    25
Name: count, dtype: int64
Selected regular SKU: SKU_00383 with 730 daily records.


In [4]:
from commerce_ai.forecasting.datasets import time_series_train_test_split

train_df, test_df = time_series_train_test_split(series_regular, test_days=30, date_col="date")
print(f"Train window: {train_df['date'].min().date()} to {train_df['date'].max().date()} ({len(train_df)} days)")
print(f"Test window:  {test_df['date'].min().date()} to {test_df['date'].max().date()} ({len(test_df)} days)")


Train window: 2024-01-01 to 2025-11-30 (700 days)
Test window:  2025-12-01 to 2025-12-30 (30 days)


In [5]:
from commerce_ai.forecasting.baselines import NaiveModel, SeasonalNaiveModel, MovingAverageModel
from commerce_ai.forecasting.exponential_smoothing import ExponentialSmoothingModel
from commerce_ai.forecasting.evaluation import ForecastEvaluator

evaluator = ForecastEvaluator()
horizon = len(test_df)
actual_units = test_df["units_sold"].values

baselines = {
    "Naive": NaiveModel(),
    "Seasonal Naive (7d)": SeasonalNaiveModel(seasonal_period=7),
    "Moving Average (7d)": MovingAverageModel(window=7),
    "Moving Average (14d)": MovingAverageModel(window=14),
    "Moving Average (30d)": MovingAverageModel(window=30),
    "Exponential Smoothing": ExponentialSmoothingModel(seasonal_periods=7),
}

baseline_results = []
forecast_series_dict = {}

for name, model in baselines.items():
    model.fit(train_df)
    out = model.predict(horizon=horizon, confidence_level=0.95)
    forecast_series_dict[name] = out.forecast_units
    
    metrics = evaluator.evaluate_slice(actual_units, out.forecast_units, out.lower_bound, out.upper_bound)
    baseline_results.append({
        "Model": name,
        "MAE": metrics["mae"],
        "RMSE": metrics["rmse"],
        "WAPE (%)": metrics["wape"],
        "Bias (%)": metrics["bias"],
        "Runtime (s)": round(out.metadata.runtime_seconds, 4) if out.metadata else 0.0,
    })

baseline_comparison_df = pd.DataFrame(baseline_results).sort_values("WAPE (%)").reset_index(drop=True)
print("Baseline Forecast Benchmark on Regular Demand Series:")
print(baseline_comparison_df.to_string(index=False))


Baseline Forecast Benchmark on Regular Demand Series:
                Model    MAE   RMSE  WAPE (%)  Bias (%)  Runtime (s)
                Naive 2.0000 2.5690     28.85      0.96       0.0003
 Moving Average (30d) 2.7111 3.1946     39.10     27.40       0.0002
  Moving Average (7d) 3.3714 3.8916     48.63     42.17       0.0002
Exponential Smoothing 3.6811 4.2516     53.09     48.87       0.0023
  Seasonal Naive (7d) 4.8333 7.2549     69.71     47.60       0.0003
 Moving Average (14d) 5.2095 5.8082     75.14     75.14       0.0002


In [6]:
from commerce_ai.forecasting.croston import CrostonModel

# Select an intermittent SKU (or lowest occurrence rate)
int_sku_candidates = intermittency.sort_values("demand_occurrence_rate")["sku_id"]
int_sku = int_sku_candidates.iloc[0]

series_int = demand_with_mask[
    (demand_with_mask["sku_id"] == int_sku) & 
    (demand_with_mask["warehouse_id"] == target_wh)
].sort_values("date").reset_index(drop=True)

train_int, test_int = time_series_train_test_split(series_int, test_days=30, date_col="date")
actual_int = test_int["units_sold"].values

croston_classic = CrostonModel(variant="classic", alpha=0.1)
croston_classic.fit(train_int)
out_classic = croston_classic.predict(horizon=len(test_int))

croston_sba = CrostonModel(variant="sba", alpha=0.1)
croston_sba.fit(train_int)
out_sba = croston_sba.predict(horizon=len(test_int))

naive_int = NaiveModel()
naive_int.fit(train_int)
out_naive_int = naive_int.predict(horizon=len(test_int))

int_benchmark = [
    {"Model": "Naive", **evaluator.evaluate_slice(actual_int, out_naive_int.forecast_units)},
    {"Model": "Croston Classic", **evaluator.evaluate_slice(actual_int, out_classic.forecast_units)},
    {"Model": "Croston SBA", **evaluator.evaluate_slice(actual_int, out_sba.forecast_units)},
]
int_benchmark_df = pd.DataFrame(int_benchmark)[["Model", "mae", "rmse", "wape", "bias"]]
int_benchmark_df.columns = ["Model", "MAE", "RMSE", "WAPE (%)", "Bias (%)"]
print(f"Intermittent Demand Benchmark ({int_sku} - Occurrence: {series_int['units_sold'].gt(0).mean():.1%}):")
print(int_benchmark_df.sort_values("WAPE (%)").to_string(index=False))


Intermittent Demand Benchmark (SKU_00046 - Occurrence: 95.3%):
          Model     MAE    RMSE  WAPE (%)  Bias (%)
    Croston SBA  4.7014  5.1885     79.24     78.80
Croston Classic  5.2337  5.6968     88.21     88.21
          Naive 24.0667 24.1716    405.62    405.62


In [7]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Subplot 1: Regular SKU
dates_test = pd.to_datetime(test_df["date"])
axes[0].plot(dates_test, actual_units, label="Actual Demand", color="black", linewidth=2.0, marker="o", markersize=3)
axes[0].plot(dates_test, forecast_series_dict["Seasonal Naive (7d)"], label="Seasonal Naive (7d)", linestyle="--", linewidth=1.5)
axes[0].plot(dates_test, forecast_series_dict["Moving Average (7d)"], label="Moving Avg (7d)", linestyle="-.", linewidth=1.5)
axes[0].plot(dates_test, forecast_series_dict["Exponential Smoothing"], label="Exponential Smoothing", linestyle=":", linewidth=2)
axes[0].set_title(f"Baseline Forecasts on Regular Demand Series ({regular_sku} @ {target_wh})", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Units Sold")
axes[0].legend(loc="upper right")

# Subplot 2: Intermittent SKU
dates_int = pd.to_datetime(test_int["date"])
axes[1].plot(dates_int, actual_int, label="Actual Intermittent Demand", color="black", linewidth=2.0, marker="o", markersize=3)
axes[1].plot(dates_int, out_classic.forecast_units, label="Croston Classic", linestyle="--", linewidth=1.5)
axes[1].plot(dates_int, out_sba.forecast_units, label="Croston SBA (Deflated)", linestyle="-.", linewidth=2.0, color="crimson")
axes[1].set_title(f"Intermittent Demand Modeling ({int_sku} @ {target_wh})", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Units Sold")
axes[1].set_xlabel("Forecast Date")
axes[1].legend(loc="upper right")

plt.tight_layout()
plot_path = workspace_dir / ".gemini" / "scratch" / "baselines_plot.png"
plt.savefig(plot_path, dpi=120)
plt.close(fig)
print(f"Forecast baseline visualization saved successfully to: {plot_path}")


Forecast baseline visualization saved successfully to: /Users/avijeet/Documents/eRetail AI Intelligence/.gemini/scratch/baselines_plot.png
